# Canonical-nacional-data
## 1. Objective and scope

Reproducible national diagnostic of notification quality, territorial/temporal coverage and
conditional eligibility for a future national canonical dataset. This is an exploratory
intermediate analysis, not the national canonical pipeline, a G1 experiment, a new H5 label,
or a modification of BETA V1. Count units are **notifications**, not deduplicated cases.

**Cloud handoff:** the epidemiological CSVs may be absent. In that case the executed notebook
reports `READY_FOR_LOCAL_EXECUTION`; no synthetic scientific evidence or zero calendar is created.
Engineering fixtures live only in unit tests.

The `09_` prefix follows the existing numbered notebook sequence. Earlier notebooks remain frozen.

## 2. Sources and methodological assumptions

Required epidemiological sources are `data/raw/dengue.csv` and `data/raw/dengue_grave.csv`.
The full territorial reference is `data/sources/Divipola-municipios.md`, a JSON document with a
`.md` suffix. The department snapshot, DANE provenance JSON and `Listados_DIVIPOLA.xlsx` are
also inventoried. Original Stage 2 manifests/contracts are checked when present; existing
pinned source identities are reused. The Excel is supplementary: its absence does not require
inventing an alternative catalogue or downloading new data.

- **D1:** accept event/notification date differences. Preserve original fields; use the existing
  `INI_SIN` day/month/year parser for aggregation. Do not replace onset with `FEC_NOT`.
- **D2:** within the selected territorial and review-period universe, no notification rows means zero.
- **D3:** complete municipality × month counts separately for event 210 and event 220, with source-row
  and absence indicators. This assumption does not apply to climate; no climate is transformed here.
- Keep all notification classifications and all possible duplicates. Event identity must agree with
  its source stream; disagreements are counted for review, not silently reclassified.
- Full-row SHA-256 fingerprints identify identical exported rows within each stream. Repeated
  `CONSECUTIVE` values and cross-stream identifier overlaps are candidate signals only; the field
  has unverified patient/episode semantics. No clinical similarity rule deletes records.

Controlling references: [field dictionary](../docs/product/stage-2/DATA-01-Priority%20Field%20Matrix.md),
[scientific decisions](../docs/product/traking-concluciones.md),
[data policy](../docs/data/data-policy.md),
[BETA01 protocol](../docs/product/experiments/EXP-BETA01/protocol.phase2.json).
Historical context, not new authority:
[recognition report](../docs/archive/experimental-eda/stage-2/DATA-01-Initial%20Recognition%20Report.md)
and [2013 investigation](../docs/archive/experimental-eda/stage-2/DATA-01-2013-date-inconsistency-investigation.md).

## 3. Configuration and loading

Run with the existing project environment and notebook extra (`.[dev,notebook]`). No new dependencies.
All adjustable parameters are centralized below. The default **review** interval follows BETA01's
`training_start` (2011-06) and stops before the reserved 2025 block. Earlier/later observed dates
remain in source/annual/monthly diagnostics and exclusion reconciliation.

This interval is an explicit diagnostic choice, not inferred from future case density and not proof
of extraction completeness. Confirm source coverage locally before using any review row for training.
The current catalogue is the fixed territorial universe, including its non-municipal areas (ANM),
whose type is preserved. Current membership does not certify historical boundaries.

Outputs stay in the ignored `data/interim/canonical-nacional-data/<execution_id>/` directory.
The execution identity binds parameters, source hashes, status and helper-code hash. Repeated runs
with the same inputs are deterministic. Scratch duplicate fingerprints are removed after scanning.

In [1]:
from pathlib import Path
import sys

# Bootstrap from repository root or notebooks/ without a machine-specific path.
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "AGENTS.md").is_file() and (path / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))

from IPython.display import Markdown, display
from sat_dengue.data.national_diagnostics import AnalysisConfig, run_analysis

CONFIG = AnalysisConfig(
    start_month="2011-06",
    end_month="2024-12",
    chunksize=100_000,
    output_dir="data/interim/canonical-nacional-data",
)
result = run_analysis(ROOT, CONFIG)
summary, tables = result["summary"], result["tables"]
display(Markdown(f"**Execution state: `{summary['status']}`**"))

def show_table(name, columns=None, limit=15):
    """Display bounded aggregate evidence, or its exact execution gap."""
    if name not in tables:
        display(Markdown(f"`{name}`: unavailable in this execution. "
                         "Inspect the source inventory and execution state."))
        return
    frame = tables[name]
    display((frame[columns] if columns else frame).head(limit))
    if len(frame) > limit:
        display(Markdown(f"Showing {limit} of {len(frame):,} aggregate rows; "
                         "the complete table is in the execution output directory."))

**Execution state: `READY_FOR_LOCAL_EXECUTION`**

## 4. Inventory and integrity

`match` means equality with a supplied local identity, not independent confirmation of an official
complete extraction. `unreferenced` means a digest was computed but no prior identity was available.
`mismatch` blocks canonical readiness for a completed diagnostic. Missing supplementary sources are
listed as limitations. Source identities are recomputed after processing before publication.

In [2]:
show_table("inventory", ["source", "available", "integrity", "bytes"], limit=10)
show_table("source_records")

,source,available,integrity,bytes
0,data/raw/dengue.csv,False,missing,
1,data/raw/dengue_grave.csv,False,missing,
2,data/sources/Listados_DIVIPOLA.xlsx,False,missing,
3,data/sources/Divipola-municipios.md,True,match,209867.0
4,data/sources/Divipola-departamentos.md,True,match,2185.0
5,data/sources/divipola-provenance.json,True,match,925.0
6,data/interim/stage2_consolidated_dataset_pipel...,False,missing,
7,data/interim/stage2_consolidated_dataset_pipel...,False,missing,


`source_records`: unavailable in this execution. Inspect the source inventory and execution state.

## 5. Territorial quality and DANE codes

Residence attribution follows `COD_DPTO_R` + `COD_MUN_R`, using the existing strict digit/padding
normalizer. Leading zeros are preserved. Current catalogue matches are valid. Well-formed unmatched
codes with a known department enter **historical or potentially homologable** triage; this label
provides no proof of historical validity. Malformed/incomplete codes are distinct from other
well-formed codes requiring review. No historical equivalence is supplied by the current catalogue,
so no automatic reassignment is applied. Raw code components are retained in aggregate discrepancy
summaries for local investigation of previously reported discrepancies.

Territorial category proportions use **all notifications in each source** as denominator. Unresolved
records remain in reconciliation. The totals distinguish observed valid codes from the full reference
universe; ANM territories must not be mistaken for legally constituted municipalities.

In [3]:
show_table("territorial_totals")
show_table("territorial_coverage")
show_table("territorial_discrepancies")

`territorial_totals`: unavailable in this execution. Inspect the source inventory and execution state.

`territorial_coverage`: unavailable in this execution. Inspect the source inventory and execution state.

`territorial_discrepancies`: unavailable in this execution. Inspect the source inventory and execution state.

## 6. Temporal coverage

Annual/monthly notification tables describe parsed symptom-onset time across all rows, including
outside-window dates; unassignable dates get their own category. Missing or unparseable onset is not
replaced. No-record months inside the explicit review calendar are zeros under D2, not missing
outcomes. Intermittency describes notifications, not surveillance failure. Notification-year aliases
and report-date differences do not override `INI_SIN`.

Tables are used for this handoff because the Cloud execution has no epidemiological observations to
plot. Inspect the complete annual, monthly and municipal summaries locally before drawing trends.

In [4]:
show_table("annual_notifications", limit=30)
show_table("temporal_coverage")
show_table("municipality_eligibility")

`annual_notifications`: unavailable in this execution. Inspect the source inventory and execution state.

`temporal_coverage`: unavailable in this execution. Inspect the source inventory and execution state.

`municipality_eligibility`: unavailable in this execution. Inspect the source inventory and execution state.

## 7. Duplicate analysis and epidemiological consistency

Full-row fingerprints cover every original column and continue across CSV chunks. Duplicate groups,
rows involved and excess rows are separate quantities. Fingerprint equality has the usual negligible
SHA-256 collision assumption. Identifier groups are stream-qualified; cross-stream overlaps are
reported separately. Absent/blank identifiers are counted, not grouped as a single duplicate identity.
No deduplication is performed.

`record_reconciliation` uses joint, mutually exclusive flag combinations so its counts sum to all
source rows. Individual flags can overlap and must not be summed independently. Unknown events,
source/event disagreements, missing essential fields and unassignable geography/onset are explicit.

In [5]:
show_table("duplicates")
show_table("event_consistency")
show_table("essential_missingness", limit=20)
show_table("record_reconciliation")

`duplicates`: unavailable in this execution. Inspect the source inventory and execution state.

`event_consistency`: unavailable in this execution. Inspect the source inventory and execution state.

`essential_missingness`: unavailable in this execution. Inspect the source inventory and execution state.

`record_reconciliation`: unavailable in this execution. Inspect the source inventory and execution state.

## 8. Municipality-month calendar with zeros

The exploratory matrix contains one row per catalogue territory and fixed review month, separate
210/220 accepted notification counts, source-row counts, `has_rows` and `zero_from_absence` flags.
A month containing rejected event rows is marked `requires_record_review`; its zero accepted count
must not be interpreted as absence of source rows. No private row copy, patient identifier, climate
imputation, endemic channel, target or training artifact is published.

Runtime checks enforce unique keys, nonnegative counts, total source reconciliation and exact
conservation of accepted notifications. Both private streams must be present and nonempty, and
some rows must be accepted, before a calendar is emitted. A missing stream never becomes zeros.

In [6]:
show_table("municipality_month", [
    "municipality_code", "year_month", "dengue_notifications_count",
    "dengue_grave_notifications_count", "dengue_notifications_count_source_rows",
    "dengue_notifications_count_zero_from_absence", "requires_record_review",
    "eligible_for_canonical_review",
], limit=12)

`municipality_month`: unavailable in this execution. Inspect the source inventory and execution state.

## 9. Conditional national eligibility

A notification is accepted for this matrix if its residence code matches DIVIPOLA, `INI_SIN` is
interpretable and inside the configured interval, and its event is 210/220 and agrees with its stream.
A catalogue territory-month is eligible **for canonical review** if it has no locally assignable
rejected event rows. Zero frequency never excludes a territory. Municipal and period summaries
count these conditional rows; they do not select based on later case counts or future outcomes.

`training_eligibility=NOT_ASSESSED` and historical territorial applicability remains `UNVERIFIED`.
Actual G1 admission additionally needs a reviewed extraction interval, historically appropriate
territorial universe, climate, feature history, temporal splits and point-in-time safeguards.
No fit-ready national sample is asserted here. H5 rules and all frozen BETA protocols are preserved.

In [7]:
show_table("municipality_eligibility")
show_table("period_eligibility")

`municipality_eligibility`: unavailable in this execution. Inspect the source inventory and execution state.

`period_eligibility`: unavailable in this execution. Inspect the source inventory and execution state.

## 10. Results and conclusions

The conclusion below comes from this execution. Missing real data yields a local-execution handoff,
not scientific validation. Successful real processing can yield only conditional viability with
limitations; unresolved historical applicability and extraction provenance preclude an unconditional
`READY_FOR_NATIONAL_CANONICAL` claim in this implementation.

In [8]:
display(Markdown(f"**Conclusion: `{summary['status']}`**\n\n"
                 f"Evidence: `{summary['evidence']}`. "
                 f"Source preservation: `{summary['source_preservation']}`."))
if summary["missing_sources"]:
    display(Markdown("Unavailable source/reference files:\n\n" +
                     "\n".join(f"- `{path}`" for path in summary["missing_sources"])))
display(Markdown(summary.get("recommendation", "Run locally with the two original private CSVs "
                 "and inspect the national diagnostics before deciding canonical viability.")))
display(Markdown(f"Execution metadata and available aggregate tables: "
                 f"`{CONFIG.output_dir}/{summary['execution_id']}/`. "
                 "This directory is ignored by Git."))

**Conclusion: `READY_FOR_LOCAL_EXECUTION`**

Evidence: `NOT_SCIENTIFICALLY_VALIDATED`. Source preservation: `PASS`.

Unavailable source/reference files:

- `data/raw/dengue.csv`
- `data/raw/dengue_grave.csv`
- `data/sources/Listados_DIVIPOLA.xlsx`
- `data/interim/stage2_consolidated_dataset_pipeline/execution_manifest.json`
- `data/interim/stage2_consolidated_dataset_pipeline/candidate_construction_contract_v1.json`

Run locally with the two original private CSVs and inspect the national diagnostics before deciding canonical viability.

Execution metadata and available aggregate tables: `data/interim/canonical-nacional-data/90933d61e1dc81be57c1/`. This directory is ignored by Git.

## 11. Limitations and recommendations for the next phase

1. Hash equality does not prove official extraction completeness or historical surveillance coverage.
2. Current DANE codes, including ANM, cannot establish historical applicability. Review unmatched codes
   against authoritative historical mappings; do not invent equivalences.
3. Accepted retrospective date differences do not demonstrate historical operational availability.
4. No-record zeros are an approved modelling assumption, not evidence of complete reporting.
5. Duplicates remain notifications; identifier semantics and 210/220 overlap are unresolved.
6. Resolve schema errors, empty streams and integrity mismatches before canonical construction.
7. Review conditional month-level flags and coverage edges, then separately define the national
   canonical contract. National climate availability and G1 training remain future work.

To reproduce locally, restore the original private files under `data/raw/`, select the project
virtual environment as kernel, and run every cell in order. Programmatic execution from repository
root: `.venv/bin/python -c 'from nbclient import NotebookClient; import nbformat; n=nbformat.read("notebooks/09_canonical_nacional_data.ipynb", as_version=4); NotebookClient(n, timeout=3600).execute()'`.
Do not commit an executed notebook containing private epidemiological outputs. Clear those outputs
before a PR; keep the full real execution in the ignored `data/` destination.

Cloud checks validate implementation and the missing-input path. Real national coverage, duplicates,
epidemiological consistency and training viability remain pending local execution.